In [ ]:
using Pkg
Pkg.activate("..")
using Distributed
using ClusterManagers
using Eliashberg
using GLMakie
println("Eliashberg loaded successfully!")

## 1D Tight-Binding model 

In [ ]:
lattice= ChainLattice(1.5) # 晶格常数 1.5 的一维链
kgrid = generate_reciprocal_lattice(lattice, 2000)

tb = TightBinding(lattice, 1.0, -0.3)

In [ ]:
dispersion_data = compute_dispersion_curve_data(tb, kgrid)
f = plot_dispersion_curves(dispersion_data.coordinates, dispersion_data.bands)
save("tb_1d_eband.png", f)
f


### Coulomb interaction

In [ ]:
interaction = ConstantInteraction(-2.5)
sc_field = BCSReducedPairing(:s_wave)

phis = range(0.0, 1.0, length=100)
Ts = range(0.1, 0.4, length=10)

kpath = generate_kpath(lattice; n_pts_per_segment=50)
q_nodes = [SVector{1, Float64}(0.0), SVector{1, Float64}(pi)]
q_labels = ["Γ", "X"]
qpath = generate_kpath(q_nodes, q_labels; n_pts_per_segment=200)
omegas = range(0.0, 5.0, length=400)
T_plasmon = 0.01


In [ ]:
# 对当前版本的包，等离激元示例直接展示裸密度响应的动态谱函数。
# 相互作用修正可以在后处理阶段进一步叠加到 χ₀(q, ω) 上。

spectral_matrix = scan_spectral_function(tb, kgrid, qpath, omegas; T=T_plasmon, η=0.02)
fig_plasmon = plot_spectral_function(qpath, omegas, spectral_matrix)
display(fig_plasmon)


In [ ]:
phase_data = compute_phase_transition_data(phis, Ts, sc_field, tb, interaction, kgrid)
plot_phase_transition(phis, Ts, phase_data.condensation_energy, phase_data.order_parameters)


In [ ]:
band_data = compute_renormalized_band_data(Ts, sc_field, tb, interaction, kgrid, kpath)
plot_renormalized_bands(Ts, kpath, band_data.bare_bands, band_data.hole_bands, band_data.particle_bands, band_data.gaps)


In [ ]:
T_val = 0.1
collective_data = compute_collective_mode_spectral_data(T_val, sc_field, tb, interaction, kgrid, kpath; eta=0.015)
plot_collective_modes(kpath, collective_data.omegas, collective_data.spectral_matrix; pair_breaking_edge=collective_data.pair_breaking_edge)


### FFLO Pairing

In [ ]:
# 定义引力相互作用 (超导是由有效吸引力驱动的，我们设 V < 0)
interaction = ConstantInteraction(-2.5)

# 设定环境参数
T_val = 0.01
h_val = 0.5 # 施加一个 Zeeman 磁场 (注意：h 必须小于裸能隙，但在 Pauli 极限附近)

# 2. 设定我们要扫描的 q 范围 (沿着 qx 方向)
q_vals = range(0.0, 1.0, length=300) # 扫描 0 到 1.0 的动量

zeeman_data = compute_zeeman_pairing_data(T_val, h_val, q_vals, tb, interaction, kgrid)
plot_zeeman_pairing_landscape(q_vals, zeeman_data.condensation_energy, zeeman_data.optimal_gaps; minimum_index=zeeman_data.minimum_index)